In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import seaborn as sns
import numpy as np
from datetime import datetime
import os
import pickle


from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import davies_bouldin_score
from sklearn.metrics import silhouette_score

In [ ]:
df_clients = pd.read_csv(os.path.join('data_client_cleaned.csv'))

In [ ]:
pd.set_option('display.max_rows', 20)

In [ ]:
df_clients

In [ ]:
df_clients.shape

In [ ]:
df_clients.isna().sum()

In [ ]:
#suppression des colonnes corrélées
df_clients.drop(['nb_total_produits','Montant_max_achats'], 
          axis=1, 
          inplace=True)

In [ ]:
df_clients.select_dtypes(['category','object'])

In [ ]:
df_clients.set_index('customer_id', inplace=True)

In [ ]:
df_clients

In [ ]:
df_clients.dtypes

In [ ]:

#df_clients['delai_moy_livraison'] = df_clients['delai_moy_livraison'].astype('timedelta[s]')
#df_clients['delai_dernier_achat'] = df_clients['delai_dernier_achat'].astype()'timedelta[s]')

In [ ]:
#Echantillon réduit
#Préparation

data = df_clients.drop(['customer_unique_id','customer_city','seller_id'], axis=1).copy()
data_reduce = data.sample(10000, random_state=1).copy()
X = data_reduce.copy()
categorical_columns = X.select_dtypes(['category','object']).columns
numerical_columns = X.select_dtypes(['int32','float64']).columns

ohe = OneHotEncoder(sparse_output=False)
ss = StandardScaler()
X[numerical_columns] = ss.fit_transform(X[numerical_columns])
ohe.fit(X[categorical_columns])

In [ ]:
X = pd.merge(X[numerical_columns], 
          pd.DataFrame(columns = ohe.get_feature_names_out().tolist(),
              data = ohe.fit_transform(X[categorical_columns])).set_index(X.index),
        left_index = True, right_index = True)

In [ ]:
#Transformation des types des catégories sorties du one hot encoding
import re
regex = re.compile(r'x\d_')
for column in X.columns:
    if regex.search(column):
        X[column] = X[column].astype('int32')

In [ ]:
X.head()

In [ ]:
#PCA transformation
pca = PCA(random_state=1)
pca.fit(X)

In [ ]:
plt.figure(figsize=(12,5))
plt.title('Somme cumulée de variance des composantes de l\'ACP')
plt.xlabel('Nombre de composantes de l\'ACP')
plt.ylabel('Variance cumulée')
plt.plot(list(range(X.shape[1])),
         np.cumsum(pca.explained_variance_ratio_))

In [ ]:
np.cumsum(pca.explained_variance_ratio_)[12]

In [ ]:
#avec 12 features on a 90 % de la variance totale

In [ ]:
colonnes_pca = X.columns

In [ ]:
from matplotlib.patches import Circle
sns.set_context("paper", font_scale=1.5) 
plt.figure(figsize=(15,15))
chaine = "Plan factoriel pour les 2 premières composantes de l'ACP"
plt.title(chaine)
plt.xlabel('Composante 1')
plt.ylabel('Composante 2')
ax = plt.gca()
ax.add_patch(Circle([0,0], radius=1, color='black', fill=False, linestyle='-', 
                    clip_on=False))
ax.set_xlim(-1,1)
ax.set_ylim(-1,1)
x_pca = pca.components_[0,:]
y_pca = pca.components_[1,:]

sns.scatterplot(x = x_pca,
                y = y_pca,
               alpha = 0.7)
plt.plot([-1,1],[0,0], color='grey', alpha=0.8)
plt.plot([0,0],[-1,1], color='grey', alpha=0.8)

i = 0
for x,y,colonne in zip(x_pca,y_pca, colonnes_pca):

    plt.annotate(colonne, 
                 (x,y), 
                 textcoords="offset points", 
                 xytext=(0,5+np.random.randint(-10,10)),
                 ha='center') 
    ax.arrow(0, 0, x, y, head_width=0, head_length=0, fc='grey', ec='grey',
             alpha=0.2)


sns.set_context("paper", font_scale=1) 

In [ ]:
x_show = pca.transform(X.select_dtypes(['float64','int32']))[:,0]

y_show = pca.transform(X.select_dtypes(['float64','int32']))[:,1]

z_show = pca.transform(X.select_dtypes(['float64','int32']))[:,2]

In [ ]:
X_pca = pca.transform(X)[:,:12]

In [ ]:
sns.set_context("paper", font_scale=1.5) 
plt.figure(figsize=(10,10))
chaine = "Plan factoriel pour les 2 premières composantes de l'ACP"
plt.title(chaine)
plt.xlabel('Composante 1')
plt.ylabel('Composante 2')
ax = plt.gca()

sns.scatterplot(x = x_show,
                y = y_show,
                #hue = hue_show,
                alpha=0.8, s=10)

sns.set_context("paper", font_scale=1) 

In [ ]:
sns.set_context("paper", font_scale=1.5) 
plt.figure(figsize=(10,10))
chaine = "Plan factoriel pour les composantes 1 et 3 de l'ACP"
plt.title(chaine)
plt.xlabel('Composante 1')
plt.ylabel('Composante 3')
ax = plt.gca()

sns.scatterplot(x = x_show,
                y = z_show,
                #hue = hue_show,
                alpha=0.8, s=10)
sns.set_context("paper", font_scale=1) 

In [ ]:
from matplotlib.patches import Circle
sns.set_context("paper", font_scale=1.5) 
plt.figure(figsize=(10,10))
chaine = "Plan factoriel pour composantes 2 et3 de l'ACP"
plt.title(chaine)
plt.xlabel('Composante 2')
plt.ylabel('Composante 3')
ax = plt.gca()

sns.scatterplot(x = y_show,
                y = z_show,
                #hue = hue_show,
                alpha=0.8, s=10)

sns.set_context("paper", font_scale=1) 

In [ ]:
#utilisons l'algorithme T-SNE pour avoir une idée en première approche du nombre de groupes homogènes

In [ ]:
tsne = TSNE(n_components=2)
T = tsne.fit_transform(X_pca.copy())

In [ ]:
plt.figure(figsize=(8,8))
plt.title('Représentation du jeu de données via T-SNE')
sns.scatterplot(x=T[:,0], y=T[:,1])

In [ ]:
#K-means
kmeans = KMeans(n_clusters=8, 
                verbose=1, 
                random_state=0,
                ).fit(X_pca)

In [ ]:
kmeans.labels_

In [ ]:
#Recherche d'un optimum du nombre de clusters

In [ ]:
inertia = {}
dict_kmeans = {}
for k in range(1,8):
    print(k)
    kmeans = KMeans(n_clusters=k,
                   verbose=1,
                   random_state=0,
                   ).fit(X_pca)
    inertia[k] = kmeans.inertia_
    dict_kmeans[k] = kmeans
    

In [ ]:
plt.figure(figsize=(8,5))
plt.title('Kmeans: Comparaison de la somme des inerties en fonction du nombre de clusters')
sns.lineplot(x=list(inertia.keys()), y=list(inertia.values())
            )

In [ ]:
#n-samples
inertia = {}
dict_kmeans = {}

nb_boucles=7
max_clusters = 8

for k in range(1,nb_boucles+1):
    print('Boucle ',k)
    for i in range(1,max_clusters+1):
        kmeans = KMeans(n_clusters=i,
                       verbose=1,
                       ).fit(X_pca)
        inertia[(k,i)] = kmeans.inertia_
        dict_kmeans[k,i] = kmeans

In [ ]:
chaine = 'Kmeans: Comparaison de la somme des inerties en fonction du\nnombre de clusters (' + str(nb_boucles) + ' itérations)'
plt.figure(figsize=(8,5))
plt.title(chaine)
for i in range(0,nb_boucles):
    index_plot = [cluster+i*(max_clusters) for cluster in list(range(0,max_clusters))][0:max_clusters]
    sous_liste = [list(inertia.values())[index] for index in index_plot]
    sns.lineplot(x=list(range(0,max_clusters)), y=sous_liste, alpha=1/nb_boucles, color='grey')

In [ ]:
#coefficient de silhouette
silhouettes_kmeans = {}
for i in range(2,max_clusters):
    silhouettes_kmeans[i] = silhouette_score(X = X_pca,
                                             labels= list(dict_kmeans[(1,i)].labels_))

In [ ]:
plt.figure(figsize=(8,6)),
plt.title('Coefficient de silhouette moyen en fonction du nombre de clusters (kmeans)')
sns.scatterplot(x = list(silhouettes_kmeans.keys()),
               y = list(silhouettes_kmeans.values()))
plt.show()

In [ ]:
for key, value in silhouettes_kmeans.items():
    if value == max(silhouettes_kmeans.values()):
        print('le coefficient de silhouette est maximal pour {} clusters (score = {})'.format(key, value))
        nb_clusters = key

In [ ]:
#indice de Davies-Bouldin
davies_bouldin_kmeans = {}
for i in range(2,max_clusters):
    davies_bouldin_kmeans[i] = davies_bouldin_score(X = X_pca,
                                             labels= list(dict_kmeans[(1,i)].labels_))

In [ ]:
plt.figure(figsize=(8,6)),
plt.title('Davies Bouldin Score moyen en fonction du nombre de clusters (kmeans)')
sns.scatterplot(x = list(davies_bouldin_kmeans.keys()),
               y = list(davies_bouldin_kmeans.values()))
plt.show()

In [ ]:
for key, value in davies_bouldin_kmeans.items():
    if value == min(davies_bouldin_kmeans.values()):
        print('le score de Davies Bouldin est minimal pour {} clusters (score = {})'.format(key, value))

In [ ]:
#représentation
plt.figure(figsize=(8,8))
plt.title('Représentation T-SNE de la séparation du jeu de données via KMeans (7 clusters)')
sns.scatterplot(x=T[:,0],
                y=T[:,1],
               hue=dict_kmeans[(1,7)].labels_,
                legend='full',
               palette=sns.color_palette("hls", 7))

In [ ]:
#DBScan
#Application
k=1
m=2
for k in range (1,7):
    for m in range (2,7):
        dbs = DBSCAN(eps=k, min_samples=m, n_jobs=-1).fit(X_pca.copy())
        plt.figure(figsize=(8,8))
        plt.title('Représentation de la séparation des données du DBSCAN via T-SNE')
        sns.scatterplot(x=T[:,0], y=T[:,1], hue=dbs.labels_)
#dbs = DBSCAN(eps = 1, min_samples=5, n_jobs=-1).fit(X_pca.copy())
#T[:,0], T[:,1], 

In [ ]:
#Représentation des variables
#plt.figure(figsize=(8,8))
#plt.title('Représentation de la séparation des données du DBSCAN via T-SNE')
#sns.scatterplot(T[:,0], T[:,1], dbs.labels_)

In [ ]:
#Clustering Hiérarchique
nb_clusters_agg = nb_clusters

In [ ]:
from sklearn import datasets
from sklearn.cluster import AgglomerativeClustering

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

In [ ]:
#CLUSTERING HIERARCHIQUE

In [ ]:
#Métrique pour le CAH
Z = linkage(X, method="ward")
pd.DataFrame(Z)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

_ = dendrogram(Z, ax=ax)

plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Number of points in node (or index of point if no parenthesis).")
plt.ylabel("Distance.")
plt.show()

In [ ]:
# les arguments p=7, truncate_mode="lastp" signifient que l'on ne va afficher que 7 clusters

fig, ax = plt.subplots(1, 1, figsize=(12, 8))

_ = dendrogram(Z, p=7, truncate_mode="lastp", ax=ax)

plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Number of points in node (or index of point if no parenthesis).")
plt.ylabel("Distance.")
plt.show()

In [ ]:
#  les arguments p=3, truncate_mode="level"signifient que l'on ne va afficher que les clusters issus de 2 noeuds.

fig, ax = plt.subplots(1, 1, figsize=(12, 8))

_ = dendrogram(Z, p=2, truncate_mode="level", ax=ax)

plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Number of points in node (or index of point if no parenthesis).")
plt.show()

In [ ]:
#  les arguments p=3, truncate_mode="level"signifient que l'on ne va afficher que les clusters issus de 3 noeuds.

fig, ax = plt.subplots(1, 1, figsize=(12, 8))

_ = dendrogram(Z, p=3, truncate_mode="level", ax=ax)

plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Number of points in node (or index of point if no parenthesis).")
plt.show()

In [ ]:
#Un meilleur partage avec 7 clusters

In [ ]:
#On instancie l'estimateur
cah = AgglomerativeClustering(n_clusters = 7, linkage="ward"
                             ).fit(X_pca.copy())
cah

In [ ]:
#on entraîne l'estimateur
cah.fit(X)

In [ ]:
cah.labels_

In [ ]:
#dd = {i:j for i,j in enumerate(list("abc"))}
#labels = [dd[i] for i in cah.labels_]
#labels[:10]

In [ ]:
X["cluster"] = cah.labels_
X.head()

In [ ]:
graph = pd.DataFrame(data = [T[:,0], T[:,1], cah.labels_]).T
graph.columns = ['X', 'Y', 'cah.labels_']

In [ ]:
graph['x_pca'] = x_show
graph['y_pca'] = y_show
graph['z_pca'] = z_show

In [ ]:
graph

In [ ]:
plt.figure(figsize=(10,10))
plt.title('Représentation de la séparation des données du CAH via T-SNE')
sns.scatterplot(x=graph['X'],
                y=graph['Y'], 
                hue = cah.labels_, 
                legend='full',
               palette=sns.color_palette("hls",7))
plt.legend(loc='upper left', labels=graph['cah.labels_'].unique())
plt.show()

In [ ]:
from matplotlib.patches import Circle
sns.set_context("paper", font_scale=1.5) 
plt.figure(figsize=(10,10))
chaine = "Plan factoriel pour les 2 premières composantes de l'ACP\
 du clustering CAH"
plt.title(chaine)
plt.xlabel('Composante 1')
plt.ylabel('Composante 2')
ax = plt.gca()

sns.scatterplot(x = graph['x_pca'], 
                y = graph['y_pca'], 
                hue = cah.labels_,
                alpha=1, s=15, legend='full')

sns.set_context("paper", font_scale=1) 

In [ ]:
from matplotlib.patches import Circle
sns.set_context("paper", font_scale=1.5) 
plt.figure(figsize=(10,10))
chaine = "Plan factoriel pour les 2 premières composantes de l'ACP \
du clustering KMeans"
plt.title(chaine)
plt.xlabel('Composante 1')
plt.ylabel('Composante 2')
ax = plt.gca()

sns.scatterplot(x = graph['x_pca'], 
                y = graph['y_pca'], 
                hue = graph['cah.labels_'],
                alpha=1, s=15, legend='full')

sns.set_context("paper", font_scale=1) 


In [ ]:
silhouette_score(X = X_pca, labels= cah.labels_)

In [ ]:
#Le silhouette score est meilleur pour k-means que pour cah

In [ ]:
#Ensemble du jeu de données

In [ ]:
#Préparation jeu de donnée complet

In [ ]:
X_tot=df_clients.drop(['customer_unique_id','customer_city','seller_id'], axis=1).copy()

categorical_columns = X_tot.select_dtypes(['category','object']).columns
numerical_columns = X_tot.select_dtypes(['int32','float64']).columns

ohe = OneHotEncoder(sparse_output=False)
ss = StandardScaler()
X_tot[numerical_columns] = ss.fit_transform(X_tot[numerical_columns])
ohe.fit(X_tot[categorical_columns])

In [ ]:
X_tot = pd.merge(X_tot[numerical_columns], 
          pd.DataFrame(columns = ohe.get_feature_names_out().tolist(),
              data = ohe.fit_transform(X_tot[categorical_columns])).set_index(X_tot.index),
        left_index = True, right_index = True)

In [ ]:
#ACP
pca_tot = PCA(random_state=1)

In [ ]:
pca_tot.fit(X_tot)

In [ ]:
plt.figure(figsize=(12,5))
plt.title('Somme cumulée de variance des composantes de l\'ACP')
plt.xlabel('Nombre de composantes de l\'ACP')
plt.ylabel('Variance cumulée')
plt.plot(list(range(X_tot.shape[1])),
         np.cumsum(pca_tot.explained_variance_ratio_))

In [ ]:
np.cumsum(pca_tot.explained_variance_ratio_)[12]

In [ ]:
X_pca_tot = pca_tot.transform(X_tot)[:,:12]

In [ ]:
colonnes_pca = X_tot.columns

In [ ]:
#Représentation des données

In [ ]:
x_show = pca_tot.transform(X_tot.select_dtypes(['float64','int32']))[:,0]

y_show = pca_tot.transform(X_tot.select_dtypes(['float64','int32']))[:,1]

z_show = pca_tot.transform(X_tot.select_dtypes(['float64','int32']))[:,2]

In [ ]:
from matplotlib.patches import Circle
sns.set_context("paper", font_scale=1.5) 
plt.figure(figsize=(10,10))
chaine = "Plan factoriel pour les 2 premières composantes de l'ACP"
plt.title(chaine)
plt.xlabel('Composante 1')
plt.ylabel('Composante 2')
ax = plt.gca()

sns.scatterplot(x = x_show,
                y = y_show,
                alpha=0.8, s=10)


i = 0

sns.set_context("paper", font_scale=1) 

In [ ]:
#KMeans
#Modèle KMeans avec le nombre de clusters optimal identifié sur le jeu de données réduit

In [ ]:
kmeans = KMeans(n_clusters=7, 
                verbose=1, 
                random_state=0
               ).fit(X_pca_tot)

In [ ]:
kmeans.labels_.shape

In [ ]:
silhouette_score(X = X_pca_tot, labels= kmeans.labels_)

In [ ]:
inertia = {}
dict_kmeans = {}
dict_silhouette = {}
for k in range(2,8):
    kmeans = KMeans(n_clusters=k,
                   verbose=1,
                   random_state=0
                  ).fit(X_pca_tot)
    inertia[k] = kmeans.inertia_
    dict_kmeans[k] = kmeans
    dict_silhouette[k] = silhouette_score(X = X_pca_tot, labels= kmeans.labels_)
    
    

In [ ]:
for k in range(2,8):
    print('Boucle {} - Silhouette score : {}'.format(k, dict_silhouette[k]))

In [ ]:
if False:
    inertia = {}
    dict_kmeans = {}
    dict_silhouette = {}
    for k in range(2,7):
        kmeans = KMeans(n_clusters=k,
                       verbose=1,
                       random_state=0,
                       n_jobs=-1).fit(X_pca_tot)
        inertia[k] = kmeans.inertia_
        dict_kmeans[k] = kmeans
        dict_silhouette[k] = silhouette_score(X = X_pca_tot, labels= kmeans.labels_)
        print('Boucle {} - Silhouette score : {}'.format(k, dict_silhouette[k]))

In [ ]:
plt.figure(figsize=(8,6)),
plt.title('Coefficient de silhouette moyen en fonction du nombre de clusters (kmeans)')
sns.scatterplot(x = list(dict_silhouette.keys()),
               y = list(dict_silhouette.values()))
plt.show()

In [ ]:
#Le coefficient de silhouette est maximal pour 2 clusters

In [ ]:
#export kmeans
filename = 'models/kmeans_7_full_dataset.obj' 
with  open(filename, 'wb') as filehandler :
    pickle.dump(dict_kmeans[7], filehandler)

In [ ]:
#export OHE
filename = 'models/ohe.obj' 
with  open(filename, 'wb') as filehandler :
    pickle.dump(ohe, filehandler)
    
#export SS
filename = 'models/ss.obj' 
with  open(filename, 'wb') as filehandler :
    pickle.dump(ss, filehandler)
    
#export modèle PCA
filename = 'models/pca_tot.obj' 
with  open(filename, 'wb') as filehandler :
    pickle.dump(pca_tot, filehandler)

In [ ]:
#import
file = open('models/kmeans_7_full_dataset.obj', 'rb')
model = pickle.load(file)

In [ ]:
#représentation des variables
nb_clusters = 8
if False:
    tsne = TSNE(n_components=2)
    T = tsne.fit_transform(X_pca_tot.copy())
    graph = pd.DataFrame(data = [T[:,0], T[:,1], model.labels_]).T
    graph.columns = ['X', 'Y', 'label']
    plt.figure(figsize=(10,10))
    plt.title('Représentation de la séparation des données du KMEANS via T-SNE')
    sns.scatterplot(x=graph['X'], y=graph['Y'], hue = graph['label'], legend='full',
                   palette=sns.color_palette("hls",8),
                    alpha=1, s=15,)
    plt.legend(loc='upper left', labels=graph['label'].unique())
    plt.show()
    
else:
        graph = pd.DataFrame(data = [model.labels_]).T
        graph.columns = ['label']

In [ ]:
graph['x_pca'] = x_show
graph['y_pca'] = y_show
graph['z_pca'] = z_show

In [ ]:
from matplotlib.patches import Circle
sns.set_context("paper", font_scale=1.5) 
plt.figure(figsize=(10,10))
chaine = "Plan factoriel pour les 2 premières composantes de l'ACP"
plt.title(chaine)
plt.xlabel('Composante 1')
plt.ylabel('Composante 2')
ax = plt.gca()

sns.scatterplot(x = graph['x_pca'], 
                y = graph['y_pca'], 
                hue = graph['label'],
                alpha=1, s=15, legend='full')

sns.set_context("paper", font_scale=1) 

In [ ]:
plt.figure(figsize=(8,6))
plt.title('Distribution du nombre d\'individus par cluster, en pourcentage')
sns.barplot(x = graph['label'].value_counts().index, 
           y=graph['label'].value_counts().values/len(graph)*100)

In [ ]:
#Caractérisation des clusters
#Représentation

In [ ]:
graph.sample(20)

In [ ]:
X2=X_tot.copy()

In [ ]:
X2.shape

In [ ]:
X2['label'] = graph['label']
group_columns = X2.groupby('label').mean()
group_columns

In [ ]:
#Répartition par feature
X_tot_2 = data.copy()
X_tot_2['label'] = model.labels_

In [ ]:
X_tot_2.head(10)

In [ ]:
for column in group_columns.columns:
    try:
        plt.figure(figsize=(7,5))
        titre = 'Moyenne de ' + str(column) + ' pour chaque cluster' 
        plt.title(titre)
        sns.barplot(x = 'label',
                    y = column,
                    data = X_tot_2, 
                    ci="sd")
        plt.show()
    except:
        print('Erreur colonne : ', column)

In [ ]:
X_tot_2.groupby('label').agg(lambda x:x.value_counts().index[0])

In [ ]:
X_tot_2.groupby('label').mean(numeric_only=True)

In [ ]:
X_tot_2.groupby('label').mean(numeric_only=True)['delai_dernier_achat'].tolist()

In [ ]:
#Caractérisation évolution dans le temps
#Idée : on pourrait réduire les données en retirant les derniers mois de données et voir si les clusters évoluent ou non : si oui cela veut dire qu’il faut actualiser sur une période de temps plus faible
#Essayer en isolant une région ou autre variable pour voir si le cluster est robuste ou non

In [ ]:
data['delai_dernier_achat'].describe()

In [ ]:
#Pour voir si le clustering est stable:

#on va retirer du jeu de données tous les clients dont le dernier achat date d'il y a moins de 5 mois avec différents pas

In [ ]:
X_tot

In [ ]:
data[data['delai_dernier_achat'] < 150].shape

In [ ]:
data[~(data['delai_dernier_achat'] < 150)].shape

In [ ]:
X_old = data[~(data['delai_dernier_achat'] < 150)].copy()

In [ ]:
categorical_columns = X_old.select_dtypes(['category','object']).columns
numerical_columns = X_old.select_dtypes(['int32','float64']).columns


X_old[numerical_columns] = ss.transform(X_old[numerical_columns])
ohe.fit(X_old[categorical_columns])

In [ ]:
X_old = pd.merge(X_old[numerical_columns], 
          pd.DataFrame(columns = ohe.get_feature_names_out().tolist(),
              data = ohe.fit_transform(X_old[categorical_columns])).set_index(X_old.index),
        left_index = True, right_index = True)

In [ ]:
X_pca_old = pca_tot.transform(X_old)[:,:12]

In [ ]:
X_old.shape

In [ ]:
#K-means
inertia = {}
dict_kmeans = {}
dict_silhouette = {}
for k in range(2,8):
    kmeans_old = KMeans(n_clusters=k,
                   verbose=1,
                   random_state=0,
                  ).fit(X_pca_old)
    inertia[k] = kmeans.inertia_
    dict_kmeans[k] = kmeans
    dict_silhouette[k] = silhouette_score(X = X_pca_old, labels= kmeans_old.labels_)
    print('Boucle {} - Silhouette score : {}'.format(k, dict_silhouette[k]))

In [ ]:
for k in range(2,8):
    print('Boucle {} - Silhouette score : {}'.format(k, dict_silhouette[k]))

In [ ]:
#Le nombre de clusters qui maximise le coefficient de silhouette est toujours 2.

In [ ]:
#Représentation
X_old.shape

In [ ]:
kmeans_old.labels_.shape

In [ ]:
X_tot_old = data[~(data['delai_dernier_achat'] < 150)].copy().copy()
X_tot_old['label'] = kmeans_old.labels_

In [ ]:
for column in group_columns.columns:
    try:
        plt.figure(figsize=(7,5))
        titre = 'Moyenne de ' + str(column) + ' pour chaque cluster' 
        plt.title(titre)
        sns.barplot(x = 'label',
                    y = column,
                    data = X_tot_old, 
                    ci="sd")
        plt.show()
    except:
        print('Erreur colonne : ', column)

In [ ]:
X_tot_old.groupby('label').agg(lambda x:x.value_counts().index[0])

In [ ]:
X_tot_old.groupby('label').mean(numeric_only=True)

In [ ]:
#ON RETIRE LES 4 DERNIERS MOIS MAINTENANT

In [ ]:
data[data['delai_dernier_achat'] < 120].shape

In [ ]:
data[~(data['delai_dernier_achat'] < 120)].shape

In [ ]:
X_old2 = data[~(data['delai_dernier_achat'] < 120)].copy()

In [ ]:
categorical_columns = X_old2.select_dtypes(['category','object']).columns
numerical_columns = X_old2.select_dtypes(['int32','float64']).columns


X_old2[numerical_columns] = ss.transform(X_old2[numerical_columns])
ohe.fit(X_old2[categorical_columns])

In [ ]:
X_old2 = pd.merge(X_old2[numerical_columns], 
          pd.DataFrame(columns = ohe.get_feature_names_out().tolist(),
              data = ohe.fit_transform(X_old2[categorical_columns])).set_index(X_old2.index),
        left_index = True, right_index = True)

In [ ]:
X_pca_old2 = pca_tot.transform(X_old2)[:,:12]

In [ ]:
X_old2.shape

In [ ]:
#K-means
inertia = {}
dict_kmeans = {}
dict_silhouette = {}
for k in range(2,8):
    kmeans_old = KMeans(n_clusters=k,
                   verbose=1,
                   random_state=0,
                  ).fit(X_pca_old2)
    inertia[k] = kmeans.inertia_
    dict_kmeans[k] = kmeans
    dict_silhouette[k] = silhouette_score(X = X_pca_old2, labels= kmeans_old.labels_)
    print('Boucle {} - Silhouette score : {}'.format(k, dict_silhouette[k]))

In [ ]:
for k in range(2,8):
    print('Boucle {} - Silhouette score : {}'.format(k, dict_silhouette[k]))

In [ ]:
X_tot_old2 = data[~(data['delai_dernier_achat'] < 120)].copy().copy()
X_tot_old2['label'] = kmeans_old.labels_

In [ ]:
for column in group_columns.columns:
    try:
        plt.figure(figsize=(7,5))
        titre = 'Moyenne de ' + str(column) + ' pour chaque cluster' 
        plt.title(titre)
        sns.barplot(x = 'label',
                    y = column,
                    data = X_tot_old2, 
                    ci="sd")
        plt.show()
    except:
        print('Erreur colonne : ', column)

In [ ]:
X_tot_old2.groupby('label').agg(lambda x:x.value_counts().index[0])

In [ ]:
X_tot_old2.groupby('label').mean(numeric_only=True)

In [ ]:
#ON RETIRE 3 MOIS MAINTENANT

In [ ]:
data[data['delai_dernier_achat'] < 90].shape

In [ ]:
data[~(data['delai_dernier_achat'] < 90)].shape

In [ ]:
X_old3 = data[~(data['delai_dernier_achat'] < 90)].copy()

In [ ]:
categorical_columns = X_old3.select_dtypes(['category','object']).columns
numerical_columns = X_old3.select_dtypes(['int32','float64']).columns


X_old3[numerical_columns] = ss.transform(X_old3[numerical_columns])
ohe.fit(X_old3[categorical_columns])

In [ ]:
X_old3 = pd.merge(X_old3[numerical_columns], 
          pd.DataFrame(columns = ohe.get_feature_names_out().tolist(),
              data = ohe.fit_transform(X_old3[categorical_columns])).set_index(X_old3.index),
        left_index = True, right_index = True)

In [ ]:
X_pca_old3 = pca_tot.transform(X_old3)[:,:12]

In [ ]:
#K-means
inertia = {}
dict_kmeans = {}
dict_silhouette = {}
for k in range(2,8):
    kmeans_old = KMeans(n_clusters=k,
                   verbose=1,
                   random_state=0,
                  ).fit(X_pca_old3)
    inertia[k] = kmeans.inertia_
    dict_kmeans[k] = kmeans
    dict_silhouette[k] = silhouette_score(X = X_pca_old3, labels= kmeans_old.labels_)
    

In [ ]:
for k in range(2,8):
    print('Boucle {} - Silhouette score : {}'.format(k, dict_silhouette[k]))

In [ ]:
X_tot_old3 = data[~(data['delai_dernier_achat'] < 90)].copy().copy()
X_tot_old3['label'] = kmeans_old.labels_

In [ ]:
for column in group_columns.columns:
    try:
        plt.figure(figsize=(7,5))
        titre = 'Moyenne de ' + str(column) + ' pour chaque cluster' 
        plt.title(titre)
        sns.barplot(x = 'label',
                    y = column,
                    data = X_tot_old3, 
                    ci="sd")
        plt.show()
    except:
        print('Erreur colonne : ', column)

In [ ]:
X_tot_old3.groupby('label').agg(lambda x:x.value_counts().index[0])

In [ ]:
X_tot_old3.groupby('label').mean(numeric_only=True)

In [ ]:
#ON RETIRE 2 mois MAINTENANT

In [ ]:
data[data['delai_dernier_achat'] < 60].shape

In [ ]:
data[~(data['delai_dernier_achat'] < 60)].shape

In [ ]:
X_old4 = data[~(data['delai_dernier_achat'] < 60)].copy()

In [ ]:
categorical_columns = X_old4.select_dtypes(['category','object']).columns
numerical_columns = X_old4.select_dtypes(['int32','float64']).columns


X_old4[numerical_columns] = ss.transform(X_old4[numerical_columns])
ohe.fit(X_old4[categorical_columns])

In [ ]:
X_old4 = pd.merge(X_old4[numerical_columns], 
          pd.DataFrame(columns = ohe.get_feature_names_out().tolist(),
              data = ohe.fit_transform(X_old4[categorical_columns])).set_index(X_old4.index),
        left_index = True, right_index = True)

In [ ]:
X_pca_old4 = pca_tot.transform(X_old4)[:,:12]

In [ ]:
#K-means
inertia = {}
dict_kmeans = {}
dict_silhouette = {}
for k in range(2,8):
    kmeans_old = KMeans(n_clusters=k,
                   verbose=1,
                   random_state=0,
                  ).fit(X_pca_old4)
    inertia[k] = kmeans.inertia_
    dict_kmeans[k] = kmeans
    dict_silhouette[k] = silhouette_score(X = X_pca_old4, labels= kmeans_old.labels_)
    

In [ ]:
for k in range(2,8):
    print('Boucle {} - Silhouette score : {}'.format(k, dict_silhouette[k]))

In [ ]:
X_tot_old4 = data[~(data['delai_dernier_achat'] < 60)].copy().copy()
X_tot_old4['label'] = kmeans_old.labels_

In [ ]:
for column in group_columns.columns:
    try:
        plt.figure(figsize=(7,5))
        titre = 'Moyenne de ' + str(column) + ' pour chaque cluster' 
        plt.title(titre)
        sns.barplot(x = 'label',
                    y = column,
                    data = X_tot_old4, 
                    ci="sd")
        plt.show()
    except:
        print('Erreur colonne : ', column)

In [ ]:
X_tot_old4.groupby('label').agg(lambda x:x.value_counts().index[0])

In [ ]:
X_tot_old4.groupby('label').mean(numeric_only=True)

In [ ]:
#Calcul du ARI Score
from sklearn.metrics.cluster import adjusted_rand_score

In [ ]:
X_tot_2

In [ ]:
X_tot_old

In [ ]:
Z1 = pd.merge(X_tot_2['label'], X_tot_old['label'], on='customer_id')
Z2 = pd.merge(X_tot_2['label'], X_tot_old2['label'], on='customer_id')
Z3 = pd.merge(X_tot_2['label'], X_tot_old3['label'], on='customer_id')
Z4 = pd.merge(X_tot_2['label'], X_tot_old4['label'], on='customer_id')
Z4

In [ ]:
adjusted_rand_score(Z1['label_x'],Z1['label_y'])

In [ ]:
adjusted_rand_score(Z2['label_x'],Z2['label_y'])

In [ ]:
adjusted_rand_score(Z3['label_x'],Z3['label_y'])

In [ ]:
adjusted_rand_score(Z4['label_x'],Z4['label_y'])